# Die beste Route für wen?

**Starter-Notebook für die Projekte 2 bis 4 in Informatik und Nachhaltigkeit (WiSe 2026/27)**

Routenplaner suchen meist die kürzeste oder schnellste Route. Je nach Person zählt aber
anderes: mit dem Rad abseits der Hauptstraßen, im Rollstuhl ohne Stufen, als blinde Person
über Ampeln mit akustischem Signal. Dieses Notebook

1. lädt das Wegenetz von Dortmund-Süd aus OpenStreetMap,
2. zeigt, wie vollständig die Angaben dort sind,
3. leitet daraus Kriterien je Wegstück ab,
4. berechnet Routen für vier Personas und
5. vergleicht sie in einer Tabelle und auf einer Karte.

Es benutzt dieselben Daten, Kriterien und Personas wie die
[Webdemo](https://ls11ae.github.io/iun-routenplanung/). Am Ende stehen Ideen, wie Sie
daraus Ihr Projekt machen.

**Voraussetzungen:** Python-Umgebung wie im README beschrieben und das Datenpaket
(`python daten/herunterladen.py`). Kein Server, kein Java.

**Wichtig:** Das hier ist ein Prototyp auf unvollständigen Daten. Die Routen sind keine
Empfehlung und keine Navigationshilfe, auch nicht für Menschen mit Behinderung.

In [1]:
import time

import pandas as pd

import routen_hilfen as r

pd.set_option("display.precision", 1)
if not (r.DATEN / "dortmund-sued-wege.osm.bz2").exists():
    raise SystemExit("Das Datenpaket fehlt. Bitte zuerst im Wurzelverzeichnis ausführen: python daten/herunterladen.py")

## 1. Wegenetz laden

`lade_netz` liest die OSM-Datei mit [OSMnx](https://osmnx.readthedocs.io) und baut daraus einen
Graphen ([NetworkX](https://networkx.org)): Knoten sind Kreuzungen und Wegenden, Kanten die
Wegstücke dazwischen. Das dauert beim ersten Mal etwa eine Minute und braucht rund 1 GB
Arbeitsspeicher; danach liegt das fertige Netz in einer Zwischendatei und lädt in Sekunden.

Vereinfachung: Alle Wege gelten in beide Richtungen, Einbahnstraßen werden nicht beachtet.

In [2]:
t = time.time()
G = r.lade_netz()              # Wegenetz Dortmund-Süd: Hauptbahnhof, Campus, Phoenix-See
r.kriterien_berechnen(G)       # Kriterien je Kante, siehe Abschnitt 3
r.hoehen_ergaenzen(G)          # Höhe je Knoten, Anstieg je Kante
print(f"{G.number_of_nodes()} Knoten, {G.number_of_edges()} gerichtete Kanten, {time.time() - t:.0f} s")

53940 Knoten, 140090 gerichtete Kanten, 200 s


Jede Kante trägt die OSM-Merkmale (*tags*) ihres Weges. So sieht eine Kante aus:

In [3]:
u, v, daten = next((u, v, d) for u, v, d in G.edges(data=True) if d.get("surface") and d.get("name"))
{k: w for k, w in daten.items() if k in r.WEG_MERKMALE or k == "length"}

{'highway': 'residential',
 'name': 'Joseph-von-Fraunhofer-Straße',
 'surface': 'asphalt',
 'smoothness': 'excellent',
 'lit': 'yes',
 'maxspeed': '30',
 'oneway': False,
 'cycleway:both': 'no',
 'sidewalk': 'both',
 'length': np.float64(8.594587997587327)}

## 2. Was steht in den Daten, und was fehlt?

Für Ihre Ergebnisse ist entscheidend, wie vollständig die Angaben sind. Die erste Tabelle
zeigt für einige Merkmale, an welchem Anteil der Wegstücke (und der Weglänge) überhaupt
etwas eingetragen ist. Die zweite Tabelle betrachtet die Fußgängerquerungen.

In [4]:
wege, querungen = r.datenqualitaet(G)
wege.style.format({"Anteil der Wegstücke": "{:.0%}", "Anteil der Länge": "{:.0%}"})

,OSM-Schlüssel,Anteil der Wegstücke,Anteil der Länge
Merkmal,,,
Belag,surface,87%,88%
Oberflächengüte,smoothness,13%,15%
Breite,width,5%,7%
Steigung,incline,4%,2%
Beleuchtung,lit,79%,80%
Tempolimit,maxspeed,19%,28%
für Rollstuhl geeignet,wheelchair,3%,2%
Gehweg an der Straße,"sidewalk, sidewalk:left, sidewalk:right, sidewalk:both",19%,27%
Radweg an der Straße,"cycleway, cycleway:left, cycleway:right, cycleway:both",11%,14%


In [5]:
querungen.style.format({"Anteil mit Angabe": "{:.0%}", "davon ja": "{:.0%}"}, na_rep="")

,bezogen auf,Anteil mit Angabe,davon ja
Angabe,,,
Art der Querung (crossing),3513 Querungen,88%,
Bodenindikatoren (tactile_paving),3513 Querungen,77%,14%
Bordstein (kerb),3513 Querungen,9%,
akustisches Signal (traffic_signals:sound),1246 Ampeln,82%,62%
Vibration (traffic_signals:vibration),1246 Ampeln,80%,63%
Anforderungstaster (button_operated),1246 Ampeln,88%,46%


**Lesehilfe:** Der Belag ist fast überall eingetragen, die Eignung für Rollstühle
(`wheelchair`) und die Steigung (`incline`) fast nirgends. Eine fehlende Angabe heißt
nicht „geeignet“ und nicht „ungeeignet“. Wie Ihr Prototyp damit umgeht, ist eine der
wichtigsten Entscheidungen im Projekt (siehe Abschnitt 6).

Welche Werte kommen bei einem Merkmal vor? Zum Beispiel beim Belag:

In [6]:
pd.Series([str(d.get("surface")) for _, _, d in G.edges(data=True)]).value_counts().head(10)

asphalt          58824
paving_stones    47612
None             18006
compacted         6308
concrete          2082
sett              1746
grass              908
fine_gravel        864
dirt               856
paved              506
Name: count, dtype: int64

## 3. Von Merkmalen zu Kriterien

`kriterien_berechnen` (in `routen_hilfen.py`) übersetzt die Merkmale in Zahlen je Kante:
Meter (etwa `treppe`: Meter auf Treppen) oder Stückzahlen (etwa `querung_ohne_ampel`).
Alle Kriterien sind *additiv*: Der Wert einer Route ist die Summe über ihre Kanten.

Die Regeln sind bewusst einfach und angreifbar, zum Beispiel:

- **Hauptstraße ohne Radweg:** `highway` ist `primary` oder `secondary` und es ist kein Radweg eingetragen.
- **Schlechter Belag:** `surface` ist Kopfsteinpflaster, Schotter, Erde, Gras oder Ähnliches.
- **Querung:** zählt an der Kante, die über einen Fußweg in einen Knoten mit `highway=crossing`
  führt; unterschieden nach Ampel, Ampel mit akustischem Signal und ohne Ampel.
- **Anstieg:** Höhenunterschied der beiden Endknoten aus dem Geländemodell DGM1, nur bergauf.

Die Tabelle zeigt, wie viel davon im ganzen Netz vorkommt.

In [7]:
personas, kriterien = r.lade_personas()
kanten = pd.DataFrame([{k: d[k] for k in r.KRITERIEN} for u, v, d in G.edges(data=True)])
pd.DataFrame({
    "Kriterium": [k["name"] for k in kriterien],
    "Einheit": [k["einheit"] for k in kriterien],
    "Summe im Netz": [kanten[k["id"]].sum() / 2 for k in kriterien],     # jede Strecke hat zwei Richtungen
    "Kanten mit Wert > 0": [(kanten[k["id"]] > 0).mean() for k in kriterien],
}, index=[k["id"] for k in kriterien]).style.format({"Summe im Netz": lambda x: f"{x:,.0f}".replace(",", "."), "Kanten mit Wert > 0": "{:.1%}"})

,Kriterium,Einheit,Summe im Netz,Kanten mit Wert > 0
laenge,Länge,m,2.164.824,100.0%
hauptstrasse,Hauptstraße ohne Radweg,m,99.352,3.4%
nebenstrasse,Sammelstraße ohne Radweg,m,54.163,1.7%
fahrbahn,Fahrbahn ohne Gehweg,m,660.571,25.1%
treppe,Treppen,m,12.885,2.7%
belag_schlecht,Schlechter Belag,m,112.464,4.1%
belag_unbekannt,Belag unbekannt,m,271.213,12.9%
anstieg,Höhenmeter bergauf,m,20.447,40.0%
steil,Steile Abschnitte,m,74.645,4.6%
bordstein,Hoher Bordstein,Stück,134,0.2%


## 4. Personas: Wer bewertet Wege wie?

Eine Persona legt fest, wie schnell jemand unterwegs ist, welche Kanten gesperrt sind
und wie stark jedes Kriterium zählt. Die Kosten einer Kante sind

$$\text{Kosten} = \text{Zeit} + \sum_k \text{Gewicht}_k \cdot \text{Kriterium}_k$$

Ein Gewicht von 0,5 Sekunden je Meter bei `hauptstrasse` heißt: Für jeden Meter
Hauptstraße ohne Radweg nimmt die Radpendlerin eine halbe Sekunde Umweg in Kauf.
Die Personas stehen in `webdemo/personas.json`; die Webdemo liest dieselbe Datei.

In [8]:
pd.DataFrame([{"Persona": p["name"], "Tempo (m/s)": p["tempo"], "gesperrt": ", ".join(p["sperren"]),
               "Gewichte": ", ".join(f"{k}: {g}" for k, g in p["gewichte"].items())}
              for p in personas.values()]).set_index("Persona").style.format({"Tempo (m/s)": "{:.1f}"}).set_properties(**{"text-align": "left"})

,Tempo (m/s),gesperrt,Gewichte
Persona,,,
Zu Fuß (Standard),1.2,fuss_verboten,
Radpendlerin,4.2,"treppe, rad_verboten","hauptstrasse: 0.5, nebenstrasse: 0.1, belag_schlecht: 0.3, anstieg: 6"
Rollstuhlfahrer,1.0,"treppe, bordstein, rollstuhl_verboten, fuss_verboten","belag_schlecht: 3, belag_unbekannt: 0.3, anstieg: 20, steil: 3, querung_ohne_ampel: 20, fahrbahn: 0.3"
Blinde Studentin,1.0,fuss_verboten,"querung_ohne_ampel: 180, querung_ampel_stumm: 60, querung_ohne_bodenindikator: 30, treppe: 2, fahrbahn: 1"


## 5. Routen berechnen und vergleichen

Start und Ziel sind Koordinaten. `naechster_knoten` sucht den nächsten Knoten, und zwar
nur unter denen, die die Persona überhaupt erreichen kann: Bahnsteige sind zum Beispiel
für den Rollstuhl oft „Inseln“ hinter Treppen. Die Route selbst berechnet der Algorithmus
von Dijkstra (`networkx.shortest_path`) mit den Kosten der Persona.

In [9]:
START, ZIEL = r.ORTE["Hauptbahnhof"], r.ORTE["Phoenix-See"]      # (Breite, Länge); eigene Orte gehen genauso

routen, zeilen = {}, {}
for p in personas.values():
    gewicht = r.gewicht_fuer(p)
    erreichbar = r.groesste_komponente(G, gewicht)
    start = r.naechster_knoten(G, *START, erlaubt=erreichbar)
    ziel = r.naechster_knoten(G, *ZIEL, erlaubt=erreichbar)
    routen[p["id"]] = r.route(G, start, ziel, gewicht)
    k = r.bewerten(G, routen[p["id"]], p)
    zeilen[p["name"]] = {"Länge (km)": k["laenge"] / 1000, "Zeit (min)": k["zeit"] / 60,
                         "Hauptstraße ohne Radweg (m)": k["hauptstrasse"], "Treppen (m)": k["treppe"],
                         "Querungen ohne Ampel": k["querung_ohne_ampel"],
                         "Ampeln mit akustischem Signal": k["querung_akustisch"],
                         "Höhenmeter": k["anstieg"]}
pd.DataFrame(zeilen).T.style.format("{:.0f}").format("{:.2f}", subset=["Länge (km)"]).format("{:.1f}", subset=["Zeit (min)"])

,Länge (km),Zeit (min),Hauptstraße ohne Radweg (m),Treppen (m),Querungen ohne Ampel,Ampeln mit akustischem Signal,Höhenmeter
Zu Fuß (Standard),5.21,72.3,823,50,4,19,76
Radpendlerin,5.85,24.4,18,0,3,9,51
Rollstuhlfahrer,5.52,91.9,315,0,3,23,61
Blinde Studentin,5.95,99.2,159,21,0,8,68


Die Karte zeigt die vier Routen. Sie unterscheiden sich in Farbe **und** Strichart
(durchgezogen: zu Fuß, gestrichelt: Rad, gepunktet: Rollstuhl, Strich-Punkt: blind).

In [10]:
liste = list(personas.values())
r.karte(G, [routen[p["id"]] for p in liste], namen=[p["name"] for p in liste],
        farben=[p["farbe"] for p in liste], striche=[p["strich"] for p in liste],
        gewichte=[r.gewicht_fuer(p) for p in liste])

## 6. Zielkonflikte und fehlende Angaben

**Schnell oder ruhig?** Mit dem Gewicht für Hauptstraßen ohne Radweg lässt sich einstellen,
wie viel Umweg die Radpendlerin für eine ruhigere Route in Kauf nimmt. Jede Zeile ist eine
andere Route. Solche gewichteten Summen finden aber nicht jeden sinnvollen Kompromiss:
In der Webdemo sehen Sie unter „Pareto-Front“, welche Routen dabei übersehen werden.

In [11]:
import copy

def variante(persona, **gewichte):
    """Kopie einer Persona mit geänderten Gewichten."""
    p = copy.deepcopy(persona)
    p["gewichte"].update(gewichte)
    return p

def route_fuer(p, start=START, ziel=ZIEL):
    gewicht = r.gewicht_fuer(p)
    erreichbar = r.groesste_komponente(G, gewicht)
    return r.route(G, r.naechster_knoten(G, *start, erlaubt=erreichbar),
                   r.naechster_knoten(G, *ziel, erlaubt=erreichbar), gewicht)

zeilen = []
for g in [0, 0.05, 0.2, 0.5, 2]:
    p = variante(personas["rad"], hauptstrasse=g)
    k = r.bewerten(G, route_fuer(p), p)
    zeilen.append({"Gewicht (s je m)": str(g), "Zeit (min)": k["zeit"] / 60, "Länge (km)": k["laenge"] / 1000,
                   "Hauptstraße ohne Radweg (m)": k["hauptstrasse"], "Höhenmeter": k["anstieg"]})
pd.DataFrame(zeilen).set_index("Gewicht (s je m)").style.format("{:.0f}").format("{:.2f}", subset=["Länge (km)"]).format("{:.1f}", subset=["Zeit (min)"])

,Zeit (min),Länge (km),Hauptstraße ohne Radweg (m),Höhenmeter
Gewicht (s je m),,,,
0,22.5,5.36,3552,64
0.05,23.4,5.59,280,54
0.2,23.5,5.61,166,54
0.5,24.4,5.85,18,51
2,24.4,5.85,18,51


**Was tun, wenn die Angabe fehlt?** Der Belag ist an gut jedem zehnten Meter unbekannt.
Der Rollstuhlfahrer kann solche Wege benutzen wie alle anderen (Gewicht 0), sie meiden
oder sie ganz sperren. Die Route ändert sich deutlich, und im strengsten Fall ist ein
Teil des Netzes gar nicht mehr erreichbar.

In [12]:
zeilen = {}
for name, p in [("unbekannt zählt nicht", variante(personas["rollstuhl"], belag_unbekannt=0)),
                ("unbekannt meiden (Voreinstellung)", personas["rollstuhl"]),
                ("unbekannt stark meiden", variante(personas["rollstuhl"], belag_unbekannt=5)),
                ("unbekannt gesperrt", {**personas["rollstuhl"], "sperren": personas["rollstuhl"]["sperren"] + ["belag_unbekannt"]})]:
    k = r.bewerten(G, route_fuer(p), p)
    zeilen[name] = {"Länge (km)": k["laenge"] / 1000, "Belag unbekannt (m)": k["belag_unbekannt"],
                    "schlechter Belag (m)": k["belag_schlecht"],
                    "erreichbare Knoten": len(r.groesste_komponente(G, r.gewicht_fuer(p))) / G.number_of_nodes()}
pd.DataFrame(zeilen).T.style.format("{:.0f}").format("{:.2f}", subset=["Länge (km)"]).format("{:.0%}", subset=["erreichbare Knoten"])

,Länge (km),Belag unbekannt (m),schlechter Belag (m),erreichbare Knoten
unbekannt zählt nicht,5.49,684,0,91%
unbekannt meiden (Voreinstellung),5.52,150,0,91%
unbekannt stark meiden,5.86,13,0,91%
unbekannt gesperrt,5.88,0,0,82%


## 7. Eine erste Wegbeschreibung

Für blinde und sehbehinderte Menschen zählt neben der Route die Beschreibung. Die
einfachste Fassung fasst aufeinanderfolgende Kanten mit gleichem Straßennamen zusammen
und nennt die Querungen. Abbiegerichtungen, Orientierungspunkte und verständliche
Formulierungen fehlen noch: Das wäre ein Teil von Projekt 4.

In [13]:
def wegbeschreibung(G, pfad, persona):
    gewicht = r.gewicht_fuer(persona)
    schritte = []
    for u, v in zip(pfad, pfad[1:]):
        d = r.beste_kante(G, u, v, gewicht)
        name = d.get("name") or "Weg ohne Namen"
        name = " / ".join(name) if isinstance(name, list) else name
        if not schritte or schritte[-1]["name"] != name:
            schritte.append({"name": name, "meter": 0.0, "querungen": []})
        schritte[-1]["meter"] += d["laenge"]
        art = {"querung_ohne_ampel": "Querung ohne Ampel", "querung_ampel_stumm": "Ampel ohne akustisches Signal",
               "querung_akustisch": "Ampel mit akustischem Signal"}
        schritte[-1]["querungen"] += [text for k, text in art.items() if d[k]]
    return schritte

for s in wegbeschreibung(G, routen["blind"], personas["blind"])[:12]:
    print(f"{s['meter']:5.0f} m  {s['name']}" + (f"  ({', '.join(s['querungen'])})" if s["querungen"] else ""))
print("…")

  491 m  Weg ohne Namen  (Ampel mit akustischem Signal)
   61 m  Brückstraße
  107 m  Helle
  194 m  Weg ohne Namen
   21 m  Schwanenwall
   10 m  Stiftstraße
  384 m  Weg ohne Namen  (Ampel mit akustischem Signal, Ampel mit akustischem Signal, Ampel mit akustischem Signal, Ampel mit akustischem Signal)
   70 m  Schwanenwall
  194 m  Weg ohne Namen  (Ampel mit akustischem Signal, Ampel mit akustischem Signal)
  291 m  Arndtstraße
  202 m  Hohenzollernstraße
  902 m  Kronprinzenstraße
…


## 8. Höhen

Die Höhen stammen aus dem Geländemodell DGM1 von Geobasis NRW, hier auf ein 5-m-Raster
gemittelt. Das Kriterium `anstieg` summiert die Höhenunterschiede bergauf von Knoten zu
Knoten. Zum Vergleich zählt `hoehenmeter` einen Anstieg erst, wenn er mehr als 3 m am
Stück beträgt (Hysterese). Bei groben Oberflächenmodellen mit Gebäuden und Bäumen, etwa
Copernicus, ist das nötig; hier liegen beide Zahlen nah beieinander.

Grenzen: Auf Brücken und in Unterführungen liefert ein Geländemodell die Höhe des Bodens
darunter oder darüber.

In [14]:
pd.DataFrame({p["name"]: {"Summe der Anstiege (m)": r.bewerten(G, routen[p["id"]], p)["anstieg"],
                           "mit Hysterese 3 m": r.hoehenmeter(G, routen[p["id"]])}
              for p in personas.values()}).T.style.format("{:.0f}")

,Summe der Anstiege (m),mit Hysterese 3 m
Zu Fuß (Standard),76,67
Radpendlerin,51,42
Rollstuhlfahrer,61,56
Blinde Studentin,68,53


## 9. Ein anderes Gebiet

Das Datenpaket enthält nur Dortmund-Süd. Andere Gebiete lädt `lade_netz_online` direkt
von OpenStreetMap (über den Overpass-Dienst). Das braucht Internet und kann einige
Minuten dauern; bitte nur kleine Gebiete laden und das Ergebnis nicht bei jedem Lauf neu
abrufen. Höhen gibt es dann nur, wenn Sie ein passendes Höhenraster haben.

In [15]:
ONLINE = False      # auf True setzen, um das Beispiel auszuführen
if ONLINE:
    H = r.lade_netz_online(rechteck=(7.405, 51.488, 7.420, 51.496))    # (West, Süd, Ost, Nord): Campus Nord
    # H = r.lade_netz_online(ort="Witten, Deutschland")               # oder ein Ort mit Namen
    r.kriterien_berechnen(H)
    print(H.number_of_nodes(), "Knoten")

## 10. Ihr Projekt: Ideen

**Für alle drei Projekte**

- Begründen Sie Ihre Kriterien und Gewichte mit Literatur und Richtlinien, statt die
  Voreinstellungen zu übernehmen. Ändern Sie dazu `kriterien_berechnen` und `personas.json`.
- Prüfen Sie Ihre Routen vor Ort in der Umgebung der Uni: Wo stimmen die Daten nicht?
- **Alle Kompromisse statt einiger:** Die gewichtete Summe findet nur einen Teil der
  Pareto-optimalen Routen (Abschnitt 6). Alle findet der exakte Label-Setting-Algorithmus der
  Webdemo (Ansicht „Pareto-Front“, Funktion `paretoFront` in `webdemo/js/kern.js`), den Sie aus
  der Vorlesung kennen. Schreiben Sie ihn nicht neu, sondern wenden Sie ihn auf Ihre eigenen
  Kriterien an:
  1. Berechnen Sie das Kriterium in `kriterien_berechnen` als Meter je Kante, zum Beispiel
     `d["unbeleuchtet"] = d["laenge"] * ("no" in _werte(d, "lit"))`, und tragen Sie es in `KRITERIEN` ein.
  2. Ergänzen Sie es in `FLAGGEN` in `export_webdemo.py` und unter `kriterien` in `webdemo/personas.json`.
  3. Führen Sie `python wegenetz/export_webdemo.py` aus und starten Sie die Webdemo lokal
     (`cd webdemo`, dann `python -m http.server`). Unter „Pareto-Front“ steht Ihr Kriterium dann
     zur Auswahl; die Ansicht zeigt auch, welche Routen eine gewichtete Summe findet und welche nicht.
  4. `python -m pytest tests` prüft, dass Python und JavaScript weiter dasselbe rechnen.

  Kriterien in Metern, die für eine ganze Kante gelten, brauchen keine Zeile JavaScript;
  Stückzahlen wie Querungen erfordern eine Erweiterung des Exports. Fragen für Ihr Projekt: Wie
  viele Pareto-optimale Routen gibt es für Ihre Kriterien, wie viele unterscheiden sich spürbar,
  und was sagen die übersehenen Kompromisse über Ihre Gewichte?

**Projekt 2: Entspannt mit dem Rad ans Ziel**

- Setzen Sie ein Stressmaß aus der Literatur um („Level of Traffic Stress“): Tempolimit,
  Zahl der Fahrspuren, Art des Radwegs. Wie viel fehlt dafür in den Daten?
- Nehmen Sie den Unfallatlas der Statistischen Ämter hinzu: Wo häufen sich Unfälle mit Radbeteiligung?
- Beachten Sie Einbahnstraßen (`oneway`, `oneway:bicycle`); bisher werden sie ignoriert.

**Projekt 3: Ohne Stufen ans Ziel**

- Unterscheiden Sie Rollstuhl, Rollator und Kinderwagen: Was ist gesperrt, was nur mühsam?
- Wie verlässlich ist eine „barrierefreie“ Route, wenn `wheelchair` an 2 % der Wege eingetragen ist?
  Vergleichen Sie vorsichtige und optimistische Annahmen wie in Abschnitt 6.
- Nehmen Sie Aufzüge hinzu (`highway=elevator`) und deren Status an Bahnhöfen (offene Schnittstelle der Deutschen Bahn).
- Wo liegen Steigungen über 6 %? Wie stark hängt das Ergebnis vom Höhenmodell ab?

**Projekt 4: Sicher über die Straße**

- Erweitern Sie die Wegbeschreibung aus Abschnitt 7 um Abbiegerichtungen und Querungen.
- „Einfachste Wege“: Bestrafen Sie Abbiegungen. Dafür reichen Kosten je Kante nicht mehr aus;
  wie lässt sich der Graph erweitern?
- Wie gut sind akustische Ampeln, Bodenindikatoren und Gehwege erfasst (Abschnitt 2)?

Bitte nennen Sie auf Ihrem Poster die Datenquellen: © OpenStreetMap-Mitwirkende (ODbL);
Höhen: DGM1, Geobasis NRW.